# Pattern Mining (Association Rules)

Mục tiêu
- Khám phá mối quan hệ giữa các yếu tố ảnh hưởng đến kết quả học tập
- Sử dụng thuật toán Apriori để tìm luật kết hợp
- Áp dụng để phát hiện nhóm học sinh có nguy cơ rớt



Quy trình 
Load → Select feature → Discretization → Apriori → Rules → Filter → Insight

In [1]:
import pandas as pd

from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import association_rules


/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


### Phần khai phá luật sử dụng toàn bộ dataset sau khi encoding.
### Không chia train/test vì mục tiêu là khám phá tri thức (data mining),
### không phải dự đoán.

In [2]:
df = pd.read_csv("../data/processed/student_encoded.csv")

df.head()

,age,Medu,Fedu,traveltime,studytime,failures,famrel,freetime,goout,Dalc,...,guardian_mother,guardian_other,schoolsup_yes,famsup_yes,paid_yes,activities_yes,nursery_yes,higher_yes,internet_yes,romantic_yes
0,18,4,4,2,2,0,4,3,4,1,...,True,False,True,False,False,False,True,True,False,False
1,17,1,1,1,2,0,5,3,3,1,...,False,False,False,True,False,False,False,True,True,False
2,15,1,1,1,2,3,4,3,2,2,...,True,False,True,False,True,False,True,True,True,False
3,15,4,2,1,3,0,3,2,2,1,...,True,False,False,True,True,True,True,True,True,True
4,16,3,3,1,2,0,4,3,2,1,...,False,False,False,True,True,False,True,True,False,False


## Select Important Features
Chỉ chọn các thuộc tính có ý nghĩa cho phân tích luật

In [3]:
df = df[["absences", "studytime", "failures", "pass"]]

df.head()


,absences,studytime,failures,pass
0,6,2,0,0
1,4,2,0,0
2,10,2,3,1
3,2,3,0,1
4,4,2,0,1


## Discretization (Rời rạc hóa dữ liệu)
Chuyển dữ liệu thành dạng nhị phân (0/1)

In [ ]:
df_rules = pd.DataFrame()

df_rules["high_absence"] = (df["absences"] > 10).astype(int)
df_rules["low_studytime"] = (df["studytime"] <= 2).astype(int)
df_rules["many_failures"] = (df["failures"] >= 1).astype(int)
df_rules["pass"] = df["pass"]

df_rules.head()


,high_absence,low_studytime,many_failures,pass
0,0,1,0,0
1,0,1,0,0
2,0,1,1,1
3,0,0,0,1
4,0,1,0,1


## Lựa chọn tham số

- min_support = 0.1: Chỉ xét các tập phổ biến xuất hiện ít nhất 10% dữ liệu
- min_confidence = 0.5: Đảm bảo các luật có độ tin cậy tối thiểu 50%
- lift > 1: Giữ lại các luật có mối quan hệ thực sự giữa các biến

## Run Apriori Algorithm


In [5]:
frequent_itemsets = apriori(
    df_rules,
    min_support=0.1,
    use_colnames=True
)

frequent_itemsets.sort_values(by="support", ascending=False).head(10)

/opt/anaconda3/lib/python3.13/site-packages/mlxtend/frequent_patterns/fpcommon.py:175: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(


,support,itemsets
1,0.767089,frozenset({low_studytime})
3,0.670886,frozenset({pass})
6,0.496203,"frozenset({low_studytime, pass})"
2,0.210127,frozenset({many_failures})
5,0.179747,"frozenset({low_studytime, many_failures})"
0,0.167089,frozenset({high_absence})
4,0.151899,"frozenset({low_studytime, high_absence})"


## Generate Association Rules


In [6]:
rules = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=0.5
)

rules.head()

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,frozenset({high_absence}),frozenset({low_studytime}),0.167089,0.767089,0.151899,0.909091,1.185119,1.0,0.023727,2.562025,0.187538,0.194175,0.609684,0.553555
1,frozenset({many_failures}),frozenset({low_studytime}),0.210127,0.767089,0.179747,0.855422,1.115154,1.0,0.018561,1.610970,0.130733,0.225397,0.379256,0.544873
2,frozenset({low_studytime}),frozenset({pass}),0.767089,0.670886,0.496203,0.646865,0.964195,1.0,-0.018427,0.931977,-0.137514,0.526882,-0.072988,0.693244
3,frozenset({pass}),frozenset({low_studytime}),0.670886,0.767089,0.496203,0.739623,0.964195,1.0,-0.018427,0.894515,-0.101393,0.526882,-0.117925,0.693244


## Filter Best Rules
 Chọn các luật có ý nghĩa:
- lift > 1 (có tương quan)
- confidence > 0.6 (độ tin cậy cao)

In [7]:
rules = rules[
    (rules['lift'] > 1) &
    (rules['confidence'] > 0.6)
]

rules.sort_values(by="lift", ascending=False).head(10)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,frozenset({high_absence}),frozenset({low_studytime}),0.167089,0.767089,0.151899,0.909091,1.185119,1.0,0.023727,2.562025,0.187538,0.194175,0.609684,0.553555
1,frozenset({many_failures}),frozenset({low_studytime}),0.210127,0.767089,0.179747,0.855422,1.115154,1.0,0.018561,1.610970,0.130733,0.225397,0.379256,0.544873




## Đánh giá luật kết hợp

Các luật được đánh giá dựa trên:

 - Support:
Mức độ phổ biến của luật trong dữ liệu

 - Confidence:
Xác suất xảy ra của vế phải khi vế trái xảy ra

 - Lift:
Đo mức độ phụ thuộc giữa các biến
 -  (lift > 1 cho thấy mối quan hệ có ý nghĩa)



## Insights từ Association Rules

1. Luật kết hợp cho thấy:
   Nếu học sinh có nhiều lần trượt (many_failures = 1)
   thì khả năng rớt môn tăng đáng kể (confidence cao, lift > 1).

2. Khi kết hợp nhiều yếu tố tiêu cực:
   - nghỉ học nhiều (high_absence = 1)
   - học ít (low_studytime = 1)
   
   → xác suất rớt tăng mạnh hơn so với từng yếu tố riêng lẻ.

3. Điều này cho thấy không chỉ một yếu tố đơn lẻ,
   mà sự kết hợp của nhiều yếu tố mới là nguyên nhân chính dẫn đến rớt môn.

4. Các luật có lift > 1 chứng minh rằng các yếu tố học tập có mối quan hệ phụ thuộc,
   không phải ngẫu nhiên.

5. Một số nhóm học sinh có thể được xác định là “high-risk group”
   dựa trên các tổ hợp đặc trưng (failures + absences + studytime).
## Ứng dụng thực tế

 - Xây dựng hệ thống cảnh báo sớm cho học sinh có nguy cơ rớt
 - Hỗ trợ giáo viên can thiệp kịp thời
 - Tối ưu chiến lược học tập dựa trên hành vi học sinh